In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.utils import resample
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
file_path = "/content/DataPreparationforPrediction.csv"  # Update with actual path
df = pd.read_csv(file_path)

# Drop rows where the target variable is missing
df = df.dropna(subset=["Whether AI has a Negative Impact on Education"])

# Encode categorical target variable
label_encoder = LabelEncoder()
df["Target"] = label_encoder.fit_transform(df["Whether AI has a Negative Impact on Education"])

# Select features (excluding categorical columns and year)
features = df.columns[1:-2]  # Exclude 'Year' and target columns

# Identify non-numeric columns and encode them
non_numeric_cols = df[features].select_dtypes(include=["object"]).columns
df[non_numeric_cols] = df[non_numeric_cols].apply(label_encoder.fit_transform)

# Handle missing values with mean imputation
imputer = SimpleImputer(strategy="mean")
df_imputed = pd.DataFrame(imputer.fit_transform(df[features]), columns=features)

# Standardize the features
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_imputed), columns=features)

# Add the target variable back
df_scaled["Target"] = df["Target"].values

# Define features and target
X = df_scaled.drop(columns=["Target"])
y = df_scaled["Target"]

# Perform bootstrap sampling
bootstrap_samples = 100
bootstrap_accuracies = []
for _ in range(bootstrap_samples):
    X_resampled, y_resampled = resample(X, y, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    bootstrap_accuracies.append(accuracy)

# Perform cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

# Predict the target for 2025
data_2025 = df_scaled.iloc[-1, :-1].values.reshape(1, -1)  # Assuming last row is 2025 data
prediction_2025 = model.predict(data_2025)
predicted_label_2025 = label_encoder.inverse_transform(prediction_2025)

# Print results
print("Bootstrap Mean Accuracy:", np.mean(bootstrap_accuracies))
print("Cross-Validation Mean Accuracy:", np.mean(cv_scores))
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("Whether AI Has a Negative Impact on Higher Education:", predicted_label_2025[0])


Bootstrap Mean Accuracy: 1.0
Cross-Validation Mean Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

         Yes       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

Whether AI Has a Negative Impact on Higher Education: Yes


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd

# Load the data
data = pd.read_csv('/content/DataPreparationforPrediction.csv')

# Encode the target variable and other categorical columns
data['Whether AI has a Negative Impact on Education'] = data['Whether AI has a Negative Impact on Education'].map({'Yes': 1, 'No': 0})
data['Whether Misuse of AI on Academic Dishonesty in Higher Education'] = data['Whether Misuse of AI on Academic Dishonesty in Higher Education'].map({'Yes': 1, 'No': 0})

# Handle missing values by filling them with 0 or another appropriate value
data = data.fillna(0)

# Drop non-numerical columns that are not needed for correlation analysis
# For example, if 'Year' is not needed, you can drop it
data = data.drop(columns=['Year'])

# Compute correlation matrix
correlation_matrix = data.corr()

# Extract correlation with the target variable
target_correlation = correlation_matrix['Whether AI has a Negative Impact on Education']

print(target_correlation)

Funds allocation to AI by Federal Government                                -0.590022
Fund allocation to Education by Federal Government                          -0.686978
Undergraduate tuition fees for Canadian citizens  (CAD)                     -0.719605
Number of students enrolled in postsecondary institutions (Millions)         0.654468
Number of people with graduate education certificate (college/university)    0.993843
Number of people with low income (bottom quantile)                           0.996091
Number of cases of data violation due to AI in Education                     0.323092
Global Number of Studies on Academic Misconduct due to AI                    0.340879
Whether Misuse of AI on Academic Dishonesty in Higher Education              1.000000
Whether AI has a Negative Impact on Education                                1.000000
Name: Whether AI has a Negative Impact on Education, dtype: float64


In [ ]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

n_trees = len(rf_model.estimators_)

# Store bootstrap samples for each tree
bootstrap_samples = []

for i in range(n_trees):
    # Get the indices of bootstrap samples used to train this tree
    bootstrap_indices = np.random.choice(range(len(X_train)), size=len(X_train), replace=True)
    bootstrap_samples.append(bootstrap_indices)

# Print the bootstrap indices for the first few trees
for i in range(min(3, n_trees)):  # Show for up to 3 trees
    print(f"Tree {i+1} Bootstrap Indices: {bootstrap_samples[i][:10]} ...")  # Show first 10 indices


Tree 1 Bootstrap Indices: [4 0 1 2 3] ...
Tree 2 Bootstrap Indices: [4 4 0 0 0] ...
Tree 3 Bootstrap Indices: [1 0 0 1 1] ...
